# Running Style Classifier

Classifies one of 5 styles from per-stride biomechanical features:
`normal | overstriding | small_steps | long_steps | too_bouncy`

**CV strategy**: leave-one-run-out — all strides from one run held out at a time.  
**Views**: perpendicular (side) and isometric (angled) runs are trained together; `view` is encoded as a feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")

FEATURE_COLS = [
    "contact_ms", "air_ms", "ratio", "pct_contact",
    "knee_angle_deg", "ankle_ahead_knee_px", "toeoff_ankle_knee_px", "cadence_spm",
    "loading_rate", "flight_arc_px", "hip_osc_px",
    "view_enc",   # perpendicular=0, isometric=1
]
TARGET    = "style"
GROUP_COL = "run_folder"

RF_PARAMS = dict(
    n_estimators     = 400,
    max_depth        = None,
    min_samples_leaf = 2,
    class_weight     = "balanced",
    random_state     = 42,
    n_jobs           = -1,
)

raw = pd.read_csv("style_stride_features.csv")
raw["view_enc"] = (raw["view"] == "isometric").astype(int)

# Impute sparse features with per-run median to recover strides lost to NaN
for col in ["flight_arc_px", "hip_osc_px"]:
    raw[col] = raw.groupby("run_folder")[col].transform(lambda x: x.fillna(x.median()))

df = raw.dropna(subset=FEATURE_COLS).reset_index(drop=True)

le = LabelEncoder()
df["label"] = le.fit_transform(df[TARGET])
classes = le.classes_

print(f"Strides after imputation + NaN drop : {len(df)}  (dropped {len(raw)-len(df)})")
print(f"Classes : {list(classes)}")
print(f"CV folds (runs) : {df[GROUP_COL].nunique()}")
print(f"\nClass distribution:")
print(df[TARGET].value_counts().to_string())

## EDA — Feature distributions by style

In [ ]:
plot_features = [
    ("contact_ms",           "Contact time (ms)"),
    ("air_ms",               "Air time (ms)"),
    ("cadence_spm",          "Cadence (steps/min)"),
    ("knee_angle_deg",       "Knee angle at landing (°)"),
    ("ankle_ahead_knee_px",  "Ankle ahead of knee at landing (px)"),
    ("toeoff_ankle_knee_px", "Ankle vs knee at toe-off (px)"),
    ("hip_osc_px",           "Hip vertical oscillation (px)"),
    ("flight_arc_px",        "Flight arc height (px)"),
]

style_order = ["normal", "overstriding", "small_steps", "too_bouncy"]
palette     = sns.color_palette("tab10", n_colors=len(style_order))

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (col, label) in zip(axes.flatten(), plot_features):
    sns.boxplot(
        data=df, x=TARGET, y=col, order=style_order,
        palette=palette, ax=ax, linewidth=0.8, fliersize=2,
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30, labelsize=8)

fig.suptitle("Feature distributions by running style", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
key_features = ["ankle_ahead_knee_px", "cadence_spm", "hip_osc_px", "knee_angle_deg"]
g = sns.pairplot(
    df[key_features + [TARGET]].dropna(),
    hue=TARGET, hue_order=style_order,
    palette=palette, plot_kws={"alpha": 0.4, "s": 15},
    diag_kind="kde",
)
g.figure.suptitle("Key feature pair-plot by style", y=1.01, fontsize=12)
plt.show()

## Leave-one-run-out Cross-Validation

In [ ]:
runs = df[GROUP_COL].unique()
y_true, y_pred = [], []
fold_results   = []

for run in runs:
    test_mask  = df[GROUP_COL] == run
    train_mask = ~test_mask

    X_train = df.loc[train_mask, FEATURE_COLS]
    y_train = df.loc[train_mask, "label"]
    X_test  = df.loc[test_mask,  FEATURE_COLS]
    y_test  = df.loc[test_mask,  "label"]

    clf = RandomForestClassifier(**RF_PARAMS)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    y_true.extend(y_test.tolist())
    y_pred.extend(preds.tolist())

    acc        = accuracy_score(y_test, preds)
    style_name = df.loc[test_mask, TARGET].iloc[0]
    view_name  = df.loc[test_mask, "view"].iloc[0]
    fold_results.append({"run": run, "style": style_name, "view": view_name, "acc": acc, "n": len(y_test)})
    print(f"  {run}  {style_name:14}  {view_name:14}  {acc*100:5.1f}%  ({len(y_test)} strides)")

y_true = np.array(y_true)
y_pred = np.array(y_pred)

overall_acc = accuracy_score(y_true, y_pred)
chance      = 1 / len(classes)
print(f"\nOverall accuracy : {overall_acc*100:.1f}%")
print(f"Chance baseline  : {chance*100:.1f}%")

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes))

## Confusion Matrices

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, mat, title in zip(
    axes,
    [cm, cm_norm],
    ["Counts", "Recall (row-normalised)"],
):
    ConfusionMatrixDisplay(mat, display_labels=classes).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=35, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)

for text in axes[1].texts:
    text.set_text(f"{float(text.get_text()):.2f}")

plt.tight_layout()
plt.show()

## Feature Importances

In [ ]:
clf_full = RandomForestClassifier(**RF_PARAMS)
clf_full.fit(df[FEATURE_COLS], df["label"])

importances = pd.Series(clf_full.feature_importances_, index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#E53935" if f == "ankle_ahead_px" else "steelblue" for f in importances.index]
importances.plot(kind="barh", ax=ax, color=colors, alpha=0.85)
ax.set_xlabel("Mean decrease in impurity")
ax.set_title("Random Forest feature importances (full dataset)")
ax.axvline(1 / len(FEATURE_COLS), color="red", ls="--", lw=0.8, label="uniform baseline")
ax.legend(fontsize=8)
for i, v in enumerate(importances):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## Per-style and Per-view Accuracy

In [ ]:
results = pd.DataFrame({"true": y_true, "pred": y_pred})
results["true_name"] = le.inverse_transform(results["true"])
results["correct"]   = results["true"] == results["pred"]

per_style = (
    results.groupby("true_name")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "strides"})
    .reindex(style_order)
)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2196F3" if v >= overall_acc else "#EF5350" for v in per_style["accuracy"]]
ax.barh(per_style.index, per_style["accuracy"] * 100, color=colors, alpha=0.85)
ax.axvline(overall_acc * 100, color="black", ls="--", lw=1, label=f"overall {overall_acc*100:.1f}%")
ax.axvline(chance * 100,       color="gray",  ls=":",  lw=1, label=f"chance  {chance*100:.1f}%")
ax.set_xlabel("Accuracy (%)")
ax.set_title("Per-style classification accuracy")
ax.legend(fontsize=8)
for i, (_, row) in enumerate(per_style.iterrows()):
    ax.text(row["accuracy"] * 100 + 0.5, i, f"{row['accuracy']*100:.1f}%  (n={row['strides']})",
            va="center", fontsize=9)
ax.set_xlim(0, 115)
plt.tight_layout()
plt.show()

In [ ]:
# Per-run fold accuracy, coloured by view
fold_df = pd.DataFrame(fold_results).sort_values("acc", ascending=False)
view_colors = {"perpendicular": "#5C6BC0", "isometric": "#26A69A"}
bar_colors  = [view_colors[v] for v in fold_df["view"]]
labels      = [f"{row['style']} ({row['view'][:4]}.)" for _, row in fold_df.iterrows()]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, fold_df["acc"] * 100, color=bar_colors, alpha=0.85)
ax.axvline(overall_acc * 100, color="black", ls="--", lw=1)
ax.axvline(chance * 100,       color="gray",  ls=":",  lw=1)
ax.set_xlabel("Accuracy (%)")
ax.set_title("Per-run fold accuracy  (blue = perpendicular, teal = isometric)")
ax.set_xlim(0, 115)
for i, (_, row) in enumerate(fold_df.iterrows()):
    ax.text(row["acc"] * 100 + 0.5, i, f"{row['acc']*100:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# Summary by view
print("\nAccuracy by view:")
for view, grp in fold_df.groupby("view"):
    print(f"  {view:14}  {grp['acc'].mean()*100:.1f}%  (±{grp['acc'].std()*100:.1f}%)")

## Save model for inference

In [ ]:
import joblib

# Train on full dataset for deployment
clf_deploy = RandomForestClassifier(**RF_PARAMS)
clf_deploy.fit(df[FEATURE_COLS], df["label"])

joblib.dump({"model": clf_deploy, "classes": classes, "features": FEATURE_COLS}, "style_model.pkl")
print("Saved style_model.pkl")
print(f"Classes : {list(classes)}")
print(f"Features: {FEATURE_COLS}")